# Define unique peptides to build Isoform Database
We will go through all canonical proteins and their isoforms if they exist. If there are isoforms, we will do in-silico tryptic digestion and compare all protein fragments to find the ones that are unique to the isoforms. This final list of tryptic peptides needs to be filtered to only contain the truly unique ones, so that there is also no match between different isoforms. The peptides are finally put into a fasta file including the positions of the fragments in the full protein to distinguish peptides coming from the same protein. The fragments are also classiefied as internal N-terminal or C-terminal, depending on their location in the protein.

# Setup
## Import and install Packages

In [ ]:
import sys
import os
import session_info

In [ ]:
import pandas as pd
import re
import ast

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/02_Isoform_Database"):
    print("WARNING: The working directory is not set to the '02_Isoform_Database'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '02_Isoform_Database' folder (\"{cwd}\").")

In [ ]:
# Data directories
mapping_dir = "../01_UniProt/data/mapping/"
unique_dir = "../01_UniProt/data/unique/"

## Read in fasta files and mapping

In [ ]:
full_proteome_unique = pd.read_csv(os.path.join(unique_dir, 'full_proteome_unique.csv'))
canonical_unique = pd.read_csv(os.path.join(unique_dir, 'canonical_unique.csv'))
fasta_canonical_unique = os.path.join(unique_dir, "unique_canonical.fasta")

iso_canonical_mapping = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping.csv'))
iso_canonical_mapping_flat = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping_flat.csv'))
iso_canonical_mapping_num = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping_num.csv'))

# Define unique isoform peptides 

In [ ]:
def digest_proteins_series(seq_series):
    """
    Performs an in silico tryptic digestion on a Pandas Series of protein sequences.
    Returns a Python set of all unique peptide strings.
    """
    # Fix 1: Use .empty to safely check if the Pandas Series is empty
    if seq_series.empty:
        return set()
        
    all_peptides = set() # Using a set automatically keeps only unique peptides
    
    for seq in seq_series:
        # Fix 2: Skip empty, null, or NaN rows safely
        if pd.isna(seq) or not isinstance(seq, str):
            continue
            
        cuts = [0]
        for i in range(len(seq) - 1):
            if seq[i] in ('K', 'R') and seq[i+1] != 'P':
                cuts.append(i + 1)
                
        if seq[-1] in ('K', 'R'):
            cuts.append(len(seq))
            
        if cuts[-1] != len(seq):
            cuts.append(len(seq))
            
        for start, end in zip(cuts[:-1], cuts[1:]):
            all_peptides.add(seq[start:end])
            
    return all_peptides

def digest_protein_with_coordinates(seq):
    """
    Performs an in silico tryptic digestion of a full protein.
    Returns a list of dictionaries containing the peptide string, 
    its 1-based start coordinate, and its 1-based end coordinate.
    """
    if not seq:
        return []
    
    cuts = [0]
    for i in range(len(seq) - 1):
        if seq[i] in ('K', 'R') and seq[i+1] != 'P':
            cuts.append(i + 1)
            
    if seq[-1] in ('K', 'R'):
        cuts.append(len(seq))
        
    if cuts[-1] != len(seq):
        cuts.append(len(seq))
        
    peptides_with_coords = []
    for start, end in zip(cuts[:-1], cuts[1:]):
        peptides_with_coords.append({
            'peptide': seq[start:end],
            'start': start + 1,  # Convert to 1-based index
            'end': end
        })
    return peptides_with_coords

In [ ]:
def extract_unique_isoform_peptides(
    mapping_df, seq_dict, min_len=7, max_len=52
):
    """Finds true unique tryptic peptides for all isoforms across the entire sequence.

    Classifies location types: if a unique peptide falls completely within
    the newly introduced C-terminal tail sequence, it is classified as 'C_term'.
    """
    results_list = []
    processed_isoforms = set()

    for _, row in mapping_df.iterrows():
        c_id = row["Entry"]
        i_id = row["Isoform_ID"]

        if i_id in processed_isoforms:
            continue

        c_seq = seq_dict.get(c_id)
        i_seq = seq_dict.get(i_id)
        if not c_seq or not i_seq:
            continue

        # 1. Get full-length digests with structural positions
        c_digest = digest_protein_with_coordinates(c_seq)
        i_digest = digest_protein_with_coordinates(i_seq)

        if not c_digest or not i_digest:
            continue

        # 2. rapid lookup set of canonical peptides
        canonical_peptide_pool = set(p["peptide"] for p in c_digest)

        # 3. Find the precise index where the alternative C-term sequence begins.
        # We find the first position from the left where characters differ, but only if the true
        # C-termini are genuinely different.
        true_c_term_has_changed = (
            i_digest[-1]["peptide"] != c_digest[-1]["peptide"]
        )

        divergence_index = None
        if true_c_term_has_changed:
            # Find the first amino acid position (1-based) where the isoform sequence breaks away from canonical
            for idx, (c_aa, i_aa) in enumerate(zip(c_seq, i_seq), start=1):
                if c_aa != i_aa:
                    divergence_index = idx
                    break
            # If canonical is a shorter prefix of the isoform, divergence starts just after canonical ends
            if divergence_index is None and len(i_seq) > len(c_seq):
                divergence_index = len(c_seq) + 1

        # 4. Sweep through all isoform peptides globally
        for p_info in i_digest:
            pep = p_info["peptide"]
            pep_len = len(pep)

            # Condition A: Peptide must be completely absent from canonical pool
            if pep not in canonical_peptide_pool:

                # Condition B: Must fit mass spectrometry length window
                if min_len <= pep_len <= max_len:

                    # 5. Classify Location Type based on structural divergence
                    is_at_isoform_start = p_info["start"] == 1

                    # Check if the unique peptide is entirely hosted inside the new C-terminal tail domain
                    is_in_alternative_c_term = (
                        divergence_index is not None
                        and p_info["start"] >= divergence_index
                    )

                    if is_at_isoform_start:
                        loc_type = "N_term"
                    elif is_in_alternative_c_term:
                        loc_type = "C_term"
                    else:
                        loc_type = "Internal"

                    results_list.append(
                        {
                            "Isoform_ID": i_id,
                            "Canonical_ID": c_id,
                            "Location_Type": loc_type,
                            "Captured_Peptide": pep,
                            "Peptide_Length": pep_len,
                            "Peptide_Coordinates_In_Isoform": f"{p_info['start']}-{p_info['end']}",
                        }
                    )

        processed_isoforms.add(i_id)

    return pd.DataFrame(results_list)

In [ ]:
# --- Execution ---
seq_lookup = full_proteome_unique.set_index('ID')['seq'].to_dict()
df_iso_peptides = extract_unique_isoform_peptides(iso_canonical_mapping_flat, seq_lookup, min_len=7, max_len=52)

In [ ]:
df_iso_peptides

In [ ]:
duplicates = df_iso_peptides[df_iso_peptides.duplicated(subset=['Captured_Peptide'], keep=False)]

In [ ]:
df_peptides_no_duplicates = df_iso_peptides[
    ~df_iso_peptides['Captured_Peptide'].isin(duplicates['Captured_Peptide'].unique())
]
df_peptides_no_duplicates

In [ ]:
#filter against alkl canonical peptides so that there are no matches against other canonical forms
canonical_peptides = digest_proteins_series(canonical_unique['seq'])

df_peptides_unique = df_peptides_no_duplicates[
    ~df_peptides_no_duplicates['Captured_Peptide'].isin(canonical_peptides)
]

df_peptides_unique

In [ ]:
# 1. Create the mask for matches
is_canonical_match = df_peptides_no_duplicates['Captured_Peptide'].isin(canonical_peptides)

# 2. Filter the dataframe and select only the relevant columns
df_removed_mappings = df_peptides_no_duplicates[is_canonical_match][['Captured_Peptide', 'Isoform_ID']]
df_removed_mappings.to_csv(('removed_peptides.csv'), index=False)

# 3. Drop duplicates to get a clean unique list of pairs
df_removed_mappings = df_removed_mappings.drop_duplicates()
df_removed_mappings['Isoform_ID'].unique() #135 isoforms, 547 peptides

import numpy as np

np.isin(df_removed_mappings['Isoform_ID'].unique(), df_peptides_unique['Isoform_ID'].unique()).sum() #73 still in DB, 62 lost

# Filter for removed isoforms that DO NOT exist in the unique dataframe
completely_removed_df = df_removed_mappings[
    ~df_removed_mappings['Isoform_ID'].isin(df_peptides_unique['Isoform_ID'])
]

# Get the unique list of these IDs
completely_removed_ids = completely_removed_df['Isoform_ID'].unique().tolist()
completely_removed_ids #9 found in te act

In [ ]:
# number of identifiable isoforms with this database
print(len(df_peptides_unique["Isoform_ID"].unique()))
print(len(df_peptides_no_duplicates["Isoform_ID"].unique()))

# Fasta Files for Database
The fasta file contains all unique tryptic peptides that can identify an isoform. The header contains the coordinates of the peptide in the isoform, so if there are several unique peptides in an isoform they are distinguishable by their position. The database contains 26257 peptides from 14206 isoforms.

In [ ]:
final_records = []

for _, row in df_peptides_unique.iterrows():
    iso_id = row['Isoform_ID']
    pep_seq = row['Captured_Peptide']
    location = row["Location_Type"]  
    coordinates = row["Peptide_Coordinates_In_Isoform"]
    
    # Add coordinates to the location part in the header
    header_id = f"sp|{iso_id}|{location}_{coordinates}"
    
    # Construct the SeqRecord
    rec = SeqRecord(Seq(pep_seq), id=header_id, description="")
    final_records.append(rec)


# Write to file
with open("Isoform_Database_only_iso.fasta", "w") as output_handle:
    SeqIO.write(final_records, output_handle, "fasta")

print(f"FASTA created with {len(final_records)} entries.")

# Numbers

In [ ]:
# 1. Get absolute counts and percentages for Location_Type
location_counts = df_peptides_unique['Location_Type'].value_counts()
location_pct = df_peptides_unique['Location_Type'].value_counts(normalize=True) * 100

# Print the breakdown
print("## Peptide Location Type Breakdown")
for loc in location_counts.index:
    print(f"* {loc}: {location_counts[loc]} peptides ({location_pct[loc]:.2f}%)")

In [ ]:
# 1. Count how many peptides each isoform has
peptides_per_isoform = df_peptides_unique['Isoform_ID'].value_counts()

# 2. Define an extended helper function to categorize the counts up to 10
def categorize_counts_extended(count):
    if count <= 10:
        return f"{count} peptide{'s' if count > 1 else ''}"
    else:
        return '11 or more peptides'

# 3. Apply categorization and get counts/percentages
isoform_distribution = peptides_per_isoform.apply(categorize_counts_extended).value_counts()
isoform_distribution_pct = peptides_per_isoform.apply(categorize_counts_extended).value_counts(normalize=True) * 100

# 4. Create the explicit order for display
order = [f"{i} peptide{'s' if i > 1 else ''}" for i in range(1, 11)] + ['11 or more peptides']

# 5. Print the breakdown
print("## Isoform Peptide-Count Distribution (Extended)")
print(f"Total Unique Isoforms: {len(peptides_per_isoform)}\n")
for cat in order:
    count = isoform_distribution.get(cat, 0)
    pct = isoform_distribution_pct.get(cat, 0.0)
    print(f"* Isoforms with {cat}: {count} ({pct:.2f}%)")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Count how many peptides each isoform has
peptides_per_isoform = df_peptides_unique['Isoform_ID'].value_counts()

# 2. Categorize counts up to 10, with 11+ as the final bucket
def categorize_counts_extended(count):
    return str(count) if count <= 10 else '11+'

counts_categorized = peptides_per_isoform.apply(categorize_counts_extended)

# 3. Define the explicit order for the X-axis
x_order = [str(i) for i in range(1, 11)] + ['11+']

# 4. Safely convert to a DataFrame by explicitly mapping index and values
# This avoids the pandas 2.0+ .reset_index() naming conflict
counts_series = counts_categorized.value_counts().reindex(x_order, fill_value=0)

plot_data = pd.DataFrame({
    'Peptides per Isoform': counts_series.index,
    'Isoform Count': counts_series.values
})

# 5. Create the plot (Using seaborn directly without plt.figure())
ax = sns.barplot(
    data=plot_data, 
    x='Peptides per Isoform', 
    y='Isoform Count'
)

# 6. Customize labels and title to make them clear and readable
ax.set_title("Distribution of Unique Peptides per Isoform", fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Number of Peptides mapped to Isoform", fontsize=12)
ax.set_ylabel("Number of Isoforms", fontsize=12)

# 7. Add text annotations on top of each bar for precision
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(
            f'{int(height)}', 
            (p.get_x() + p.get_width() / 2., height), 
            ha='center', 
            va='bottom', 
            xytext=(0, 3), 
            textcoords='offset points', 
            fontsize=9,
            fontweight='semibold'
        )

# 8. Clean layout borders and save the high-resolution file
sns.despine()
plt.tight_layout()
plt.savefig("peptide_number_distribution.png", dpi=300)

In [ ]:
peptides_per_isoform

In [ ]:
iso_canonical_mapping_num

In [ ]:
# 1. Clean up the 'Isoforms' column so they are real Python lists, not strings
def safely_convert_to_list(val):
    if pd.isna(val) or val == '':
        return []
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return []
    return []

iso_canonical_mapping_num['Isoforms'] = iso_canonical_mapping_num['Isoforms'].apply(safely_convert_to_list)

# 2. Build the clean peptide count mapping dictionary
pep_count_dict = df_peptides_unique.groupby('Isoform_ID').size().to_dict()

# 3. Sum counts across both the Canonical ID and the now-genuine Isoforms list
def sum_peptides_final(row):
    total = 0
    
    # Check Canonical_Isoform_ID
    canonical_id = str(row['Canonical_Isoform_ID']).strip() if pd.notna(row['Canonical_Isoform_ID']) else None
    if canonical_id and canonical_id in pep_count_dict:
        total += pep_count_dict[canonical_id]
        
    # Check alternative Isoforms (now guaranteed to be actual lists!)
    for iso_id in row['Isoforms']:
        clean_iso_id = str(iso_id).strip()
        if clean_iso_id in pep_count_dict:
            total += pep_count_dict[clean_iso_id]
            
    return total

# 4. Generate the column
iso_canonical_mapping_num['Num_Peptides'] = iso_canonical_mapping_num.apply(sum_peptides_final, axis=1)

In [ ]:
iso_canonical_mapping_num[iso_canonical_mapping_num['Num_Peptides'] >= 100]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Filter out entries with 0 peptides to focus the plot on detected data 
# (Optional: remove this filter if you want to include the 0-peptide baselines)
plot_df = iso_canonical_mapping_num[iso_canonical_mapping_num['Num_Peptides'] > 0].copy()

# 2. To avoid a cluttered x-axis if there are rare entries with 6+ isoforms,
# group everything from 4 alternative isoforms and above into a '4+' category.
def bin_isoforms(count):
    return str(count) if count < 10 else '10+'

plot_df['Isoform_Group'] = plot_df['Num_Isoforms'].apply(bin_isoforms)
x_order = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10+']

# 3. Create the plot (Using Seaborn directly)
# We use a log scale for the Y-axis because peptide counts often span orders of magnitude
ax = sns.boxplot(
    data=plot_df,
    x='Isoform_Group',
    y='Num_Peptides',
    order=x_order,
    fliersize=3,       
    linewidth=1.5
)

# 4. Customize labels and title
ax.set_title("Unique Peptide Yield by Number of Alternative Isoforms", fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Number of Alternative Isoforms", fontsize=12)
ax.set_ylabel("Number of Unique Peptides (Log Scale)", fontsize=12)
ax.set_yscale('log') # Log scale prevents huge outliers from compressing the boxes

# 5. Add a grid behind the boxes for easier reading across the log scale
ax.grid(axis='y', linestyle='--', alpha=0.5)

# 6. Clean layout borders and save the high-resolution file
sns.despine()
plt.tight_layout()
plt.savefig("isoforms_vs_peptides_distribution.png", dpi=300)